# Find ocean-icechunks datasets by place and time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/stac-geoparquet-query.ipynb)

Which Icechunk stores cover a place and a period? We search the STAC geoparquet catalog built
by `stac-geoparquet-create` with [rustac](https://github.com/stac-utils/rustac), then open
the matches in xarray with [xpystac](https://github.com/stac-utils/xpystac).

**Run `stac-geoparquet-create` first, in the same session.** It writes the catalog file that
this notebook reads.

In [ ]:
# xpystac from GitHub: release 0.5.0 only reads storage:schemes from a Collection, and our
# catalog has them on each Item. Switch to the release once a newer one is out.
%pip install -q "icechunk>=2.2" "xarray>=2026.7" "zarr>=3.4" rustac pystac "xpystac @ git+https://github.com/stac-utils/xpystac.git@main"

In [ ]:
from pathlib import Path

PARQUET = "ocean-icechunks-items.parquet"
if not Path(PARQUET).exists():
    raise FileNotFoundError(f"{PARQUET} not found: run stac-geoparquet-create first, in this same session")

## 1. The whole catalog

With no filter, a search returns every item: one per Icechunk store.

In [ ]:
import rustac

everything = await rustac.search(PARQUET)
print(len(everything), "stores")
for item in everything:
    p = item["properties"]
    print(f"{item['id']:36s} {p.get('start_datetime', 'no time   ')[:10]}  {p.get('end_datetime', '')[:10]}  {list(item['assets'])}")

## 2. Search by place and time

Oahu, Hawaii, during January 2026. The `bbox` is west, south, east, north in degrees. Items whose
box crosses the antimeridian (the North Pacific ocean heat content) are matched correctly.

The time range of an item covers all of its datasets, so an item can match even when only some of
its datasets cover the dates.

In [ ]:
BBOX = [-158.5, 20.5, -157.5, 21.5]
DATETIME = "2026-01-01/2026-02-01"

hits = await rustac.search(PARQUET, bbox=BBOX, datetime=DATETIME)
print(f"{len(hits)} of {len(everything)} stores intersect the box during {DATETIME}\n")
for item in hits:
    p = item["properties"]
    print(f"{item['id']}\n  {p['start_datetime']} .. {p['end_datetime']}\n  datasets: {list(item['assets'])}")

assert 0 < len(hits) < len(everything), "expected some, but not all, stores"

Several stores match, but not all. The reanalyses ended before 2026, the ocean heat content
stores for other ocean basins do not cover Hawaii, and so on. Try your own place and dates by
changing `BBOX` and `DATETIME`.

## 3. Open the matches

The assets are `application/vnd.zarr+icechunk`, so xpystac can open them: pass the asset to
`xr.open_dataset` with `engine="stac"`. It reads the bucket from the item's `storage:schemes`.

xpystac always opens the root of a store, and the ocean heat content stores keep their data in
groups. For those assets, which carry our `icechunk:group` field, we open the store ourselves.

In [ ]:
import icechunk
import pystac
import xarray as xr


def open_asset(asset):
    if "icechunk:group" not in asset.extra_fields:
        return xr.open_dataset(asset, engine="stac")
    scheme = asset.owner.properties["storage:schemes"][asset.extra_fields["storage:refs"][0]]
    storage = icechunk.s3_storage(
        bucket=scheme["bucket"], prefix=asset.href.split(f"{scheme['bucket']}/")[1],
        region=scheme["region"], anonymous=scheme.get("anonymous", False))
    store = icechunk.Repository.open(storage).readonly_session(asset.extra_fields["version"]).store
    return xr.open_zarr(store, group=asset.extra_fields["icechunk:group"], consolidated=False)


for item in map(pystac.Item.from_dict, hits):
    for key, asset in item.assets.items():
        ds = open_asset(asset)
        print(f"{item.id} / {key}: {len(ds.data_vars)} variables, {dict(ds.sizes)}")

Each `ds` is a lazy xarray Dataset, so nothing is downloaded until you compute something.
Pick one and select the point and period you searched for, for example
`ds.sel(time=slice("2026-01-01", "2026-02-01"))`.